In [0]:
%run ../config

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StringType
from pyspark.sql.window import Window


# Ensure Customers Silver Table Exists
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {silver_catalog}.customers_clean (
    customer_id INT NOT NULL,
    email STRING NOT NULL,
    full_name STRING NOT NULL,
    status INT,
    gender STRING,
    birth_date DATE,
    region STRING,
    city STRING,
    town STRING,
    district STRING,
    address STRING,
    last_modified_date TIMESTAMP,
    effective_start_date TIMESTAMP,
    effective_end_date TIMESTAMP,
    is_current BOOLEAN
)
""")


# Read Bronze Data
df = (
    spark.read.table(f"{bronze_catalog}.customers")
    .filter(F.col("user_id").isNotNull())
    .filter(F.col("user_name").contains("@"))
)


# Silver Layer Transformation   
df = (
    # Deduplicate records
    df.withColumn(
        "row_num",
        F.row_number().over(
            Window.partitionBy("user_id")
            .orderBy(F.col("last_modified_date").desc())
        )
    )
    .filter(F.col("row_num") == 1)
    .drop("row_num")

    # Gender normalisation
    .withColumn("user_gender",
        F.when(F.upper(F.col("user_gender")) == 'F', 'Female')
         .when(F.upper(F.col("user_gender")) == 'M', 'Male')
         .otherwise('n/a')     
    )

    # Birthdate validation
    .withColumn("user_birth_date",
        F.when(F.col("user_birth_date") > F.current_date(), None)
        .otherwise(F.col("user_birth_date"))            
    )
)

# Clean and standardise all string columns
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        if field.name == "user_name":
            df = df.withColumn(field.name, F.trim(F.col(field.name)))
        else:
            df = df.withColumn(field.name, F.initcap(F.trim(F.col(field.name))))

# Rename columns
RENAME_MAP = {
    "user_id": "customer_id",
    "user_name": "email",
    "name_surname": "full_name",
    "user_gender": "gender",
    "user_birth_date": "birth_date",
    "address_text": "address"
}
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)


# Final schema + metadata
silver_df = (
    df.select("customer_id", "email", "full_name", "status", "gender", "birth_date", "region", "city",
              "town", "district", "address", "last_modified_date")
    .withColumn("effective_start_date", F.col("last_modified_date"))
    .withColumn("effective_end_date", F.lit(None).cast("timestamp"))
    .withColumn("is_current", F.lit(True))
)


# Merge into silver table (SCD Type2)
silver_df.createOrReplaceTempView("staging_customers")

spark.sql(f"""
MERGE INTO {silver_catalog}.customers_clean t
USING staging_customers s
ON t.customer_id = s.customer_id
AND t.is_current = true

WHEN MATCHED AND (
    t.email <> s.email OR
    t.full_name <> s.full_name OR
    t.status <> s.status OR
    t.gender <> s.gender OR
    t.birth_date <> s.birth_date OR
    t.region <> s.region OR
    t.city <> s.city OR
    t.town <> s.town OR
    t.district <> s.district OR
    t.address <> s.address
)
THEN UPDATE SET
  t.is_current = false,
  t.effective_end_date = current_timestamp()

WHEN NOT MATCHED
THEN INSERT (
  customer_id,
  email,
  full_name,
  status,
  gender,
  birth_date,
  region,
  city,
  town,
  district,
  address,
  last_modified_date,
  effective_start_date,
  effective_end_date,
  is_current
)
VALUES (
  s.customer_id,
  s.email,
  s.full_name,
  s.status,
  s.gender,
  s.birth_date,
  s.region,
  s.city,
  s.town,
  s.district,
  s.address,
  s.last_modified_date,
  current_timestamp(),
  NULL,
  true
)
""")


# Validate that the Silver table was written successfully
count = spark.table(f"{silver_catalog}.customers_clean").count()

# Ensure table is not empty
assert count > 0, "customers_clean table is empty"

# Log result to job output
print(f"customers_clean validation passed: {count:,} records")